# loading Fact_Hotel_Booking Related Data from DeltaLake

In [0]:
# ============================================================================
# CONFIGURATION - Update these paths as needed
# ============================================================================

S3_PATHS = {
    "flight_bookings_silver": "s3://travel-analytics-bronze/delta/silver/flight_bookings/",
    "dim_aircraft": "s3://travel-analytics-bronze/delta/gold/Dim_Aircraft/",
    "dim_date": "s3://travel-analytics-bronze/delta/gold/Dim_Date/",
    "dim_time": "s3://travel-analytics-bronze/delta/gold/Dim_Time/",
    "dim_airline": "s3://travel-analytics-bronze/delta/gold/Dim_Airline/", 
    "dim_airport": "s3://travel-analytics-bronze/delta/gold/Dim_Airport/", 
    "dim_customer": "s3://travel-analytics-bronze/delta/gold/Dim_Customer/", 
    "fact_flight_booking_output": "s3://travel-analytics-bronze/delta/gold/Fact_Flight_Booking/"
}
# ============================================================================
# STEP 1: Read Data from Delta Lake
# ============================================================================

print("="*80)
print("READING DATA FROM DELTA LAKE")
print("="*80)

# Read Flight Bookings (Silver layer)
flight_bookings_silver_df = spark.read.format("delta").load(S3_PATHS["flight_bookings_silver"])
print(f"Flight Bookings Silver: {flight_bookings_silver_df.count():,} records")

# Reading Aircraft Dimension (Gold layer)
dim_aircraft_df = spark.read.format("delta").load(S3_PATHS["dim_aircraft"])
print(f"Dim_Aircraft: {dim_aircraft_df.count():,} records")

# Reading Date Dimension (Gold layer)
date_df = spark.read.format("delta").load(S3_PATHS["dim_date"])
print(f"Dim_Date: {date_df.count():,} records")

# Reading Time Dimension (Gold layer)
time_df = spark.read.format("delta").load(S3_PATHS["dim_time"])
print(f"Dim_Time: {time_df.count():,} records")

# Reading Airline Dimension (Gold layer)
dim_airline_df = spark.read.format("delta").load(S3_PATHS["dim_airline"])
print(f"Dim_Airline: {dim_airline_df.count():,} records")

# Reading Airport Dimension (Gold layer)
dim_airport_df = spark.read.format("delta").load(S3_PATHS["dim_airport"])
print(f"Dim_Airport: {dim_airport_df.count():,} records")

# Reading Customer Dimension (Gold layer)
customer_df = spark.read.format("delta").load(S3_PATHS["dim_customer"])
print(f"Dim_Customer: {customer_df.count():,} records")

READING DATA FROM DELTA LAKE
Flight Bookings Silver: 7,648 records
Dim_Aircraft: 246 records
Dim_Date: 8,035 records
Dim_Time: 86,400 records
Dim_Airline: 1,251 records
Dim_Airport: 105 records
Dim_Customer: 1,000 records


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ========================================================
# STEP 0: Base fact table
# ========================================================
fact_df = flight_bookings_silver_df

# ========================================================
# STEP 1: Join Aircraft Dimension
# ========================================================
print("Joining Aircraft Dimension...")
fact_df = fact_df.join(
    dim_aircraft_df.select(
        F.col("Dim_Aircraft_SK"),
        F.col("Aircraft_ID_BK")
    ),
    fact_df.Aircraft_Id == F.col("Aircraft_ID_BK"),
    "left"
)

# ========================================================
# STEP 2: Join Airline Dimension
# ========================================================
print("Joining Airline Dimension...")
fact_df = fact_df.join(
    dim_airline_df.select(
        F.col("Dim_Airline_SK"),
        F.col("Airline_ID_BK")
    ),
    fact_df.Airline_Id == F.col("Airline_ID_BK"),
    "left"
)

# ========================================================
# STEP 3: Join Source Airport Dimension
# ========================================================
print("Joining Source Airport Dimension...")
airport_src = dim_airport_df.select(
    F.col("Dim_Airport_SK").alias("Airport_Src_SK"),
    F.col("Airport_ID_BK").alias("Airport_Src_ID_BK")
)
fact_df = fact_df.join(
    airport_src,
    fact_df.Airport_Src == F.col("Airport_Src_ID_BK"),
    "left"
).drop("Airport_Src_ID_BK")

# ========================================================
# STEP 4: Join Destination Airport Dimension
# ========================================================
print("Joining Destination Airport Dimension...")
airport_dst = dim_airport_df.select(
    F.col("Dim_Airport_SK").alias("Airport_Dst_SK"),
    F.col("Airport_ID_BK").alias("Airport_Dst_ID_BK")
)
fact_df = fact_df.join(
    airport_dst,
    fact_df.Airport_Dst == F.col("Airport_Dst_ID_BK"),
    "left"
).drop("Airport_Dst_ID_BK")

# ========================================================
# STEP 5: Join Customer Dimension
# ========================================================
print("Joining Customer Dimension...")
fact_df = fact_df.join(
    customer_df.select(
        F.col("Dim_Customer_SK"),
        F.col("Customer_ID_BK")
    ),
    fact_df.Customer_Id == F.col("Customer_ID_BK"),
    "left"
)

# ========================================================
# STEP 6: Join Departure Date Dimension
# ========================================================
print("Joining Departure Date Dimension...")
date_departure = date_df.select(
    F.col("Dim_Date_SK").alias("Departure_Date_SK"),
    F.col("Full_Date").alias("Departure_Full_Date")
)
fact_df = fact_df.join(
    date_departure,
    fact_df.Departure_Date == F.col("Departure_Full_Date"),
    "left"
).drop("Departure_Full_Date")

# ========================================================
# STEP 7: Join Departure Time Dimension
# ========================================================
print("Joining Departure Time Dimension...")
time_departure = time_df.select(
    F.col("Time_Dim_SK").alias("Departure_Time_SK"),
    F.col("Full_Time").alias("Departure_Full_Time")
)
fact_df = fact_df.join(
    time_departure,
    fact_df.Departure_Time == F.col("Departure_Full_Time"),
    "left"
).drop("Departure_Full_Time")

# ========================================================
# STEP 8: Join Arrival Date Dimension
# ========================================================
print("Joining Arrival Date Dimension...")
date_arrival = date_df.select(
    F.col("Dim_Date_SK").alias("Arrival_Date_SK"),
    F.col("Full_Date").alias("Arrival_Full_Date")
)
fact_df = fact_df.join(
    date_arrival,
    fact_df.Arrival_Date == F.col("Arrival_Full_Date"),
    "left"
).drop("Arrival_Full_Date")

# ========================================================
# STEP 9: Join Arrival Time Dimension
# ========================================================
print("Joining Arrival Time Dimension...")
time_arrival = time_df.select(
    F.col("Time_Dim_SK").alias("Arrival_Time_SK"),
    F.col("Full_Time").alias("Arrival_Full_Time")
)
fact_df = fact_df.join(
    time_arrival,
    fact_df.Arrival_Time == F.col("Arrival_Full_Time"),
    "left"
).drop("Arrival_Full_Time")

# ========================================================
# STEP 10: Join Booking Date Dimension
# ========================================================
print("Joining Booking Date Dimension...")
date_booking = date_df.select(
    F.col("Dim_Date_SK").alias("Booking_Date_SK"),
    F.col("Full_Date").alias("Booking_Full_Date")
)
fact_df = fact_df.join(
    date_booking,
    fact_df.Booking_Date == F.col("Booking_Full_Date"),
    "left"
).drop("Booking_Full_Date")

# ========================================================
# STEP 11: Join Booking Time Dimension
# ========================================================
print("Joining Booking Time Dimension...")
time_booking = time_df.select(
    F.col("Time_Dim_SK").alias("Booking_Time_SK"),
    F.col("Full_Time").alias("Booking_Full_Time")
)
fact_df = fact_df.join(
    time_booking,
    fact_df.Booking_Time == F.col("Booking_Full_Time"),
    "left"
).drop("Booking_Full_Time")

# ========================================================
# STEP 12: Drop duplicate columns if exist
# ========================================================
cols = fact_df.columns
seen = set()
final_cols = []
for c in cols:
    if c not in seen:
        final_cols.append(c)
        seen.add(c)
fact_df = fact_df.select(*final_cols)

# ========================================================
# STEP 13: Generate Surrogate Key
# ========================================================
print("Generating Flight_Booking_SK...")
window_spec = Window.orderBy(F.monotonically_increasing_id())
fact_df = fact_df.withColumn(
    "Flight_Booking_SK",
    F.row_number().over(window_spec)
)

# ========================================================
# STEP 14: Select Final Columns
# ========================================================
print("Selecting final columns...")
fact_flight_booking = fact_df.select(
    # Primary Key
    F.col("Flight_Booking_SK"),

    # Business Keys (Composite Natural Key: trip_id, flight_number, departure_date)
    F.col("Trip_Id").alias("Trip_ID_BK"),
    F.col("Flight_Number").alias("Flight_Number_BK"),
    F.col("Departure_Date").alias("Departure_Date_BK"),

    # Foreign Keys
    F.col("Dim_Customer_SK").alias("Customer_Dim_FK"),
    F.col("Dim_Airline_SK").alias("Airline_Dim_FK"),
    F.col("Dim_Aircraft_SK").alias("Aircraft_Dim_FK"),
    F.col("Airport_Src_SK").alias("Airport_Src_FK"),
    F.col("Airport_Dst_SK").alias("Airport_Dst_FK"),
    F.col("Booking_Date_SK").alias("Booking_Date_FK"),
    F.col("Booking_Time_SK").alias("Booking_Time_FK"),
    F.col("Departure_Date_SK").alias("Departure_Date_FK"),
    F.col("Departure_Time_SK").alias("Departure_Time_FK"),
    F.col("Arrival_Date_SK").alias("Arrival_Date_FK"),
    F.col("Arrival_Time_SK").alias("Arrival_Time_FK"),

    # Measures
    F.col("Flight_Duration"),
    F.col("Price").alias("Ticket_Price"),
    F.col("Discount_Amount"),
    F.col("Final_Ticket_Price"),

    # Degenerate Dimensions
    F.col("Seat_Number"),
    F.col("Travel_Class"),
    F.col("Payment_Method"),
    F.col("Booking_Status")
)

# ========================================================
# STEP 15: Validate
# ========================================================
fact_flight_booking.printSchema()
fact_flight_booking.display()

Joining Aircraft Dimension...
Joining Airline Dimension...
Joining Source Airport Dimension...
Joining Destination Airport Dimension...
Joining Customer Dimension...
Joining Departure Date Dimension...
Joining Departure Time Dimension...
Joining Arrival Date Dimension...
Joining Arrival Time Dimension...
Joining Booking Date Dimension...
Joining Booking Time Dimension...
Generating Flight_Booking_SK...
Selecting final columns...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


root
 |-- Flight_Booking_SK: integer (nullable = false)
 |-- Trip_ID_BK: integer (nullable = true)
 |-- Flight_Number_BK: string (nullable = true)
 |-- Departure_Date_BK: date (nullable = true)
 |-- Customer_Dim_FK: long (nullable = true)
 |-- Airline_Dim_FK: long (nullable = true)
 |-- Aircraft_Dim_FK: long (nullable = true)
 |-- Airport_Src_FK: long (nullable = true)
 |-- Airport_Dst_FK: long (nullable = true)
 |-- Booking_Date_FK: integer (nullable = true)
 |-- Booking_Time_FK: integer (nullable = true)
 |-- Departure_Date_FK: integer (nullable = true)
 |-- Departure_Time_FK: integer (nullable = true)
 |-- Arrival_Date_FK: integer (nullable = true)
 |-- Arrival_Time_FK: integer (nullable = true)
 |-- Flight_Duration: string (nullable = true)
 |-- Ticket_Price: double (nullable = true)
 |-- Discount_Amount: double (nullable = true)
 |-- Final_Ticket_Price: double (nullable = true)
 |-- Seat_Number: string (nullable = true)
 |-- Travel_Class: string (nullable = true)
 |-- Payment_Meth

Flight_Booking_SK,Trip_ID_BK,Flight_Number_BK,Departure_Date_BK,Customer_Dim_FK,Airline_Dim_FK,Aircraft_Dim_FK,Airport_Src_FK,Airport_Dst_FK,Booking_Date_FK,Booking_Time_FK,Departure_Date_FK,Departure_Time_FK,Arrival_Date_FK,Arrival_Time_FK,Flight_Duration,Ticket_Price,Discount_Amount,Final_Ticket_Price,Seat_Number,Travel_Class,Payment_Method,Booking_Status
1,1826,DE1930,2018-12-31,540,205,28,28,67,20181219,110000,20181231,193000,20190101,2300,04:53,717.0,55.0,662.0,28C,BUSINESS,CREDIT_CARD,COMPLETED
2,349,DL0730,2018-12-31,414,286,26,28,76,20181230,63000,20181231,73000,20181231,210500,13:35,947.0,47.0,900.0,5G,ECONOMY,CREDIT CARD,COMPLETED
3,2543,U21600,2018-12-31,708,1168,15,38,75,20181226,104500,20181231,160000,20181231,181500,02:15,527.0,52.0,475.0,17J,BUSINESS,PAYPAL,COMPLETED
4,3945,TS0715,2018-12-31,334,1111,34,47,98,20181223,60000,20181231,71500,20181231,175100,10:36,1131.0,20.0,1111.0,26B,BUSINESS,CREDIT CARD,COMPLETED
5,1983,4O1615,2018-12-31,613,1108,90,90,14,20181130,154500,20181231,161500,20181231,172500,01:10,448.0,20.0,428.0,18B,BUSINESS,DEBIT CARD,COMPLETED
6,1331,FR1945,2018-12-31,586,404,167,28,83,20181130,170000,20181231,194500,20181231,205300,01:08,446.0,20.0,426.0,44H,BUSINESS,CREDIT CARD,COMPLETED
7,2516,AR1045,2018-12-31,79,241,115,34,81,20181226,143000,20181231,104500,20181231,122500,01:40,84.0,8.0,76.0,7D,ECONOMY,DEBIT_CARD,COMPLETED
8,3657,2N0630,2018-12-31,197,112,229,81,47,20181201,43000,20181231,63000,20181231,73100,01:01,37.0,3.0,34.0,1D,ECONOMY,PAYPAL,COMPLETED
9,3970,TK0630,2018-12-31,968,293,94,38,10,20181223,50000,20181231,63000,20181231,74600,01:16,455.0,20.0,435.0,26G,BUSINESS,CREDIT CARD,COMPLETED
10,3899,TK0815,2018-12-31,804,293,69,97,58,20181230,10000,20181231,81500,20181231,190700,10:52,751.0,37.55,713.45,3B,ECONOMY,PAYPAL,COMPLETED


In [0]:
output_path = S3_PATHS["fact_flight_booking_output"]
print(f"Writing to: {output_path}")

fact_flight_booking.write \
    .format("delta") \
    .mode("overwrite") \
    .save(output_path)

print(f"Successfully wrote {fact_flight_booking.count():,} records to Delta Lake")
print("="*80)

Writing to: s3://travel-analytics-bronze/delta/gold/Fact_Flight_Booking/


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Successfully wrote 7,648 records to Delta Lake
